In [ ]:
!apt-get update -y
!apt-get install -y ffmpeg
!pip install requests librosa tqdm

import os
import requests
import zipfile
import subprocess
from pathlib import Path
import shutil
from google.colab import files
import librosa
from tqdm import tqdm

public_key = "YANDEX DISK PUBLIC LINK"

ZIP_PATH = "/content/input.zip"
INPUT_DIR = "/content/input_audio"
OUTPUT_DIR = "/content/output_audio"
LOG_FILE = "/content/processing_log.txt"
ZIP_OUT = "/content/1_non-urmi_unlabeled.zip"

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Getting download link from Yandex Disk...")

api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download"
r = requests.get(api_url, params={"public_key": public_key})
r.raise_for_status()

download_url = r.json()["href"]

print("Downloading archive...")
!wget -q -O "{ZIP_PATH}" "{download_url}"

print("Unzipping...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(INPUT_DIR)

print("Extraction done.")

def convert_audio(input_path, output_path):
    cmd = [
        "ffmpeg",
        "-y",
        "-i", input_path,
        "-ar", "16000",      # 16 kHz
        "-ac", "1",          # mono
        "-sample_fmt", "s16",# 16-bit PCM
        output_path
    ]

    result = subprocess.run(
        cmd,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE
    )

    return result.returncode == 0


def is_valid_audio(file_path, min_duration=0.3):
    try:
        duration = librosa.get_duration(filename=str(file_path))
        return duration >= min_duration
    except:
        return False


SUPPORTED_EXTENSIONS = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac"}

input_files = [p for p in Path(INPUT_DIR).rglob("*") if p.is_file()]

converted = 0
skipped = 0
failed = 0

log_lines = []

print("Processing audio files...")

for file_path in tqdm(input_files):

    if file_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        skipped += 1
        continue

    if not is_valid_audio(file_path):
        log_lines.append(f"INVALID AUDIO: {file_path}")
        skipped += 1
        continue

    relative_path = file_path.relative_to(INPUT_DIR)
    output_path = Path(OUTPUT_DIR) / relative_path

    output_path = output_path.with_suffix(".wav")

    output_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        success = convert_audio(str(file_path), str(output_path))

        if not success:
            log_lines.append(f"FFMPEG FAILED: {file_path}")
            failed += 1
        else:
            converted += 1

    except Exception as e:
        log_lines.append(f"ERROR {file_path}: {str(e)}")
        failed += 1


with open(LOG_FILE, "w") as f:
    for line in log_lines:
        f.write(line + "\n")

print("Zipping results...")

shutil.make_archive(
    ZIP_OUT.replace(".zip", ""),
    'zip',
    OUTPUT_DIR
)

print("\nDONE")
print("Converted:", converted)
print("Skipped:", skipped)
print("Failed:", failed)
print("Log saved to:", LOG_FILE)

files.download(ZIP_OUT)